# Лабораторна робота 3

## Перевірка умов теореми 5.2 про існування центра

Розглядаємо стаціонарну систему звичайних диференціальних рівнянь першого порядку з квадратичною правою частиною:

$$
\dot x = y + bx^2 + (2c+\alpha)xy + dy^2,
\qquad
\dot y = -x + ax^2 + (2b+\beta)xy + cy^2.
$$

За теоремою 5.2 нульовий стан рівноваги є центром, якщо виконується хоча б одна з шести алгебраїчних умов. Їх і перевіримо

In [5]:
# %pip install sympy numpy

import math
import numpy as np
import sympy as sp
from IPython.display import Markdown, display

a = 1
b = sp.Rational(1, 5)
c = sp.Rational(-3, 10)
d = sp.Rational(2, 5)
alpha = 0
beta = 0

coeffs = {
    "a": sp.sympify(a),
    "b": sp.sympify(b),
    "c": sp.sympify(c),
    "d": sp.sympify(d),
    "alpha": sp.sympify(alpha),
    "beta": sp.sympify(beta),
}
coeffs

{'a': 1, 'b': 1/5, 'c': -3/10, 'd': 2/5, 'alpha': 0, 'beta': 0}

## Допоміжні функції

Умови 1, 3, 4, 5 перевіряються прямою підстановкою. Для умови 2 використовуємо еквівалентну форму через існування числа $k$, щоб не ділити на нуль у вироджених випадках. Для умови 6 спочатку виконуємо поворот координат, після чого виділяємо нові коефіцієнти $a_1,b_1,c_1,d_1,\beta_1$.

In [6]:
TOL = 1e-10


def is_zero(expr, tol=TOL):
    """Перевірка рівності нулю для exact sympy або числового виразу."""
    expr = sp.simplify(expr)
    if expr == 0:
        return True
    try:
        return abs(float(sp.N(expr))) <= tol
    except (TypeError, ValueError):
        return False


def all_zero(expressions, tol=TOL):
    return all(is_zero(expr, tol) for expr in expressions)


def real_roots_of_poly(poly, var, tol=TOL):
    roots = []
    for root in sp.nroots(poly):
        if abs(sp.im(root)) <= tol:
            roots.append(float(sp.re(root)))
    return roots


def check_condition_1(a, b, c, d, alpha, beta):
    exprs = [a + c, b + d]
    return all_zero(exprs), exprs, "a+c=0, b+d=0"


def condition_2_candidates(a, b, c, d, alpha, beta, tol=TOL):
    """Шукаємо k з рівностей alpha=beta*k, b+d=(a+c)*k і кубічного рівняння."""
    k = sp.symbols("k", real=True)
    linear_eqs = [alpha - beta * k, b + d - (a + c) * k]
    cubic = a * k**3 - (3 * b + alpha) * k**2 + (3 * c + beta) * k - d

    candidates = []
    if not is_zero(beta):
        candidates.append(sp.simplify(alpha / beta))
    if not is_zero(a + c):
        candidates.append(sp.simplify((b + d) / (a + c)))

    if not candidates:
        if all_zero([alpha, b + d]):
            candidates = [sp.N(root) for root in real_roots_of_poly(cubic, k, tol)]
        else:
            return []

    good = []
    for cand in candidates:
        if all_zero([eq.subs(k, cand) for eq in linear_eqs] + [cubic.subs(k, cand)], tol):
            if not any(is_zero(cand - old, tol) for old in good):
                good.append(sp.simplify(cand))
    return good


def check_condition_2(a, b, c, d, alpha, beta):
    k_values = condition_2_candidates(a, b, c, d, alpha, beta)
    text = "існує k: alpha/beta=(b+d)/(a+c)=k і виконується кубічне рівняння"
    return len(k_values) > 0, k_values, text


def check_condition_3(a, b, c, d, alpha, beta):
    exprs = [alpha, beta]
    return all_zero(exprs), exprs, "alpha=0, beta=0"


def check_condition_4(a, b, c, d, alpha, beta):
    exprs = [a, c, beta]
    return all_zero(exprs), exprs, "a=0, c=0, beta=0"


def check_condition_5(a, b, c, d, alpha, beta):
    exprs = [b + d, alpha, beta + 5 * a + 5 * c, a * c + 2 * a**2 + d**2]
    nondegenerate = not is_zero(a + c)
    text = "b+d=0, alpha=0, beta+5a+5c=0, ac+2a^2+d^2=0, a+c!=0"
    return all_zero(exprs) and nondegenerate, exprs + [a + c], text


def rotated_coefficients(a, b, c, d, alpha, beta):
    """Повертає коефіцієнти після повороту з tan(phi)=-(a+c)/(b+d)."""
    if is_zero(b + d):
        return None

    phi = math.atan2(-float(sp.N(a + c)), float(sp.N(b + d)))
    cos_phi = sp.N(math.cos(phi), 16)
    sin_phi = sp.N(math.sin(phi), 16)

    u, v = sp.symbols("u v", real=True)
    x = cos_phi * u - sin_phi * v
    y = sin_phi * u + cos_phi * v

    F1 = y + b * x**2 + (2 * c + alpha) * x * y + d * y**2
    F2 = -x + a * x**2 + (2 * b + beta) * x * y + c * y**2

    # G = R^T F(R(u,v)). Лінійна частина зберігає вигляд (v, -u).
    G1 = sp.expand(cos_phi * F1 + sin_phi * F2)
    G2 = sp.expand(-sin_phi * F1 + cos_phi * F2)

    b1 = sp.expand(G1).coeff(u, 2).coeff(v, 0)
    d1 = sp.expand(G1).coeff(v, 2).coeff(u, 0)
    mixed1 = sp.expand(G1).coeff(u, 1).coeff(v, 1)

    a1 = sp.expand(G2).coeff(u, 2).coeff(v, 0)
    c1 = sp.expand(G2).coeff(v, 2).coeff(u, 0)
    mixed2 = sp.expand(G2).coeff(u, 1).coeff(v, 1)

    alpha1 = sp.simplify(mixed1 - 2 * c1)
    beta1 = sp.simplify(mixed2 - 2 * b1)

    return {
        "phi": phi,
        "a1": sp.N(a1, 12),
        "b1": sp.N(b1, 12),
        "c1": sp.N(c1, 12),
        "d1": sp.N(d1, 12),
        "alpha1": sp.N(alpha1, 12),
        "beta1": sp.N(beta1, 12),
        "G1": sp.N(G1, 8),
        "G2": sp.N(G2, 8),
    }


def check_condition_6(a, b, c, d, alpha, beta):
    rot = rotated_coefficients(a, b, c, d, alpha, beta)
    if rot is None:
        return False, [b + d], "не застосовується: b+d=0"

    a1 = rot["a1"]
    b1 = rot["b1"]
    c1 = rot["c1"]
    d1 = rot["d1"]
    beta1 = rot["beta1"]
    exprs = [
        a1 + c1,
        beta1,
        a1 + 5 * b1 + 5 * d1,
        b1 * a1 + 2 * a1**2 + d1**2,
    ]
    text = "після повороту: a1+c1=0, beta1=0, a1+5b1+5d1=0, b1*a1+2a1^2+d1^2=0"
    return all_zero(exprs), exprs, text


def check_all(coeffs):
    a = coeffs["a"]
    b = coeffs["b"]
    c = coeffs["c"]
    d = coeffs["d"]
    alpha = coeffs["alpha"]
    beta = coeffs["beta"]
    checks = [
        ("1",) + check_condition_1(a, b, c, d, alpha, beta),
        ("2",) + check_condition_2(a, b, c, d, alpha, beta),
        ("3",) + check_condition_3(a, b, c, d, alpha, beta),
        ("4",) + check_condition_4(a, b, c, d, alpha, beta),
        ("5",) + check_condition_5(a, b, c, d, alpha, beta),
        ("6",) + check_condition_6(a, b, c, d, alpha, beta),
    ]
    return checks

In [7]:
checks = check_all(coeffs)

rows = ["| Умова | Виконується | Що перевіряється | Значення / деталі |", "|---:|:---:|---|---|"]
for number, ok, details, description in checks:
    if isinstance(details, list):
        details_text = ", ".join([str(sp.N(item, 8)) for item in details])
    else:
        details_text = str(details)
    rows.append(f"| {number} | {'так' if ok else 'ні'} | {description} | `{details_text}` |")

display(Markdown("\n".join(rows)))

if any(ok for _, ok, _, _ in checks):
    true_conditions = [number for number, ok, _, _ in checks if ok]
    display(Markdown(
        "**Висновок:** за теоремою 5.2 нульовий стан рівноваги є центром. "
        f"Виконані умови: {', '.join(true_conditions)}."
    ))
else:
    display(Markdown(
        "**Висновок:** жодна з шести умов теореми 5.2 не виконалась. "
        "Отже, за цією теоремою не можна зробити висновок, що нульовий стан є центром."
    ))

| Умова | Виконується | Що перевіряється | Значення / деталі |
|---:|:---:|---|---|
| 1 | ні | a+c=0, b+d=0 | `0.70000000, 0.60000000` |
| 2 | ні | існує k: alpha/beta=(b+d)/(a+c)=k і виконується кубічне рівняння | `` |
| 3 | так | alpha=0, beta=0 | `0, 0` |
| 4 | ні | a=0, c=0, beta=0 | `1.0000000, -0.30000000, 0` |
| 5 | ні | b+d=0, alpha=0, beta+5a+5c=0, ac+2a^2+d^2=0, a+c!=0 | `0.60000000, 0, 3.5000000, 1.8600000, 0.70000000` |
| 6 | ні | після повороту: a1+c1=0, beta1=0, a1+5b1+5d1=0, b1*a1+2a1^2+d1^2=0 | `0.91110792, -1.5265567e-16, -0.20608393, 0.92541176` |

**Висновок:** за теоремою 5.2 нульовий стан рівноваги є центром. Виконані умови: 3.

## до умови 6

Умова 6 вимагає попереднього повороту координат. У ноутбуці використано матрицю

$$
R=\begin{pmatrix}\cos\varphi&-\sin\varphi\\ \sin\varphi&\cos\varphi\end{pmatrix},
\qquad
\varphi=\operatorname{atan2}(-(a+c),\,b+d).
$$

Якщо $F(x,y)$ — початкове векторне поле, то в нових координатах $(x_1,y_1)$ поле обчислюється як

$$
G(x_1,y_1)=R^T F(R(x_1,y_1)).
$$

Після цього з квадратичної частини $G$ виділяються коефіцієнти $a_1,b_1,c_1,d_1,\alpha_1,\beta_1$ і перевіряються рівності з пункту 6 теореми.

In [8]:
rot = rotated_coefficients(**coeffs)
if rot is None:
    display(Markdown("Для умови 6 поворот не виконувався, бо $b+d=0$."))
else:
    display(Markdown(f"$\\varphi \approx {rot['phi']:.6f}$ рад."))
    display({key: value for key, value in rot.items() if key not in ("G1", "G2")})

$\varphi pprox -0.862170$ рад.

{'phi': -0.8621700546672264,
 'a1': 0.498940052983,
 'b1': 0.357935255401,
 'c1': 0.412167869855,
 'd1': -0.498940052983,
 'alpha1': -1.82221584568,
 'beta1': -1.52655665886e-16}